# Demucs (source separation)

**Domain:** Speech & Audio · **recommended addition** · **runnable:** yes

A refresher on **Demucs** — Meta's open-source music source separation model that
splits a finished stereo mix back into its stems (**drums, bass, vocals, other**).
This notebook teaches the *principle* behind it (time–frequency masking) with tiny
CPU-only examples, then shows the real Demucs call shape gated behind a download.

## 1. What & Why

**What.** Demucs (*Deep Extractor for Music Sources*) is a neural network that takes a
mixed audio track and reconstructs the individual instrument **stems**. The current
default, **Hybrid Transformer Demucs (`htdemucs`)**, separates a song into four stems:
`drums`, `bass`, `vocals`, and `other`. It is the de-facto open-source state of the art
for music separation and is what powers many "AI stem splitter" / "vocal remover" tools.

**The problem it solves.** Once instruments are mixed and mastered into a single waveform,
their energy overlaps in both time and frequency — you cannot simply filter them apart.
Source separation learns to *un-mix*: estimate each source given only the sum.

**Reach for it when** you need stems for remixing, karaoke (vocal removal), sample
extraction, music education, pre-processing for transcription/lyric alignment, or as a
front-end that cleans up one source before a downstream model.

**Don't reach for it when** you want *speech* enhancement/denoising (use a speech model
like a speech-enhancement net or `df`/DeepFilterNet), real-time low-latency separation
(Demucs is offline and heavy), or you already have the stems (e.g. multitrack sessions).

## 2. Mental Model

Think of a finished song as a **smoothie**. Mixing blended the strawberries (vocals),
banana (bass), and ice (drums) into one liquid. Source separation is the model that has
tasted thousands of smoothies *and* their original fruits, so it can look at a new
smoothie and reconstruct "what the strawberry part must have sounded like."

Mechanically, classic separation works in the **spectrogram** (time × frequency grid):
the model predicts a **mask** — a value in [0, 1] per time–frequency bin saying *what
fraction of this bin's energy belongs to source X* — and multiplies it against the mix.
Demucs goes further with a **hybrid** design: it processes the signal **both** in the
waveform (time) domain *and* the spectrogram (frequency) domain, fuses them, and uses a
Transformer in the bottleneck to model long-range structure. The masking demo below is
the conceptual core; Demucs is a learned, far more powerful version of it.

## 3. Key Concepts

- **Stem** — one isolated source track. `htdemucs` outputs four: drums, bass, vocals, other.
- **Mask (Ideal Ratio Mask, IRM)** — per-TF-bin weight in [0, 1]; multiply the mix
  spectrogram by it to recover a source. The "ideal" mask (computed from ground truth)
  is the upper bound a masking model chases. Demucs predicts waveforms directly, but
  masking is the mental model for *why* separation is possible.
- **STFT / spectrogram** — Short-Time Fourier Transform: slices audio into overlapping
  windows and gives the complex frequency content per window. Separation happens here
  because sources are sparser and more distinguishable in TF than in raw samples.
- **Hybrid (time + frequency) model** — Demucs runs parallel temporal and spectral
  branches and merges them, getting the best of waveform fidelity and spectral structure.
- **SI-SDR (Scale-Invariant Signal-to-Distortion Ratio)** — the standard quality metric
  in dB; higher is better. Scale-invariant means a loudness difference isn't penalized.
- **Shifts / overlap / segment** — inference knobs: `shifts` averages several
  time-shifted predictions (slower, cleaner), `overlap` blends chunk boundaries,
  `segment` caps chunk length to fit memory.
- **Model variants** — `htdemucs` (default, 4-stem, hybrid transformer),
  `htdemucs_ft` (fine-tuned, slower, better), `htdemucs_6s` (adds `guitar` + `piano`),
  `mdx_extra` (older, no transformer).

## 4. Setup

Demucs is a `pip` install but pulls in **PyTorch** and downloads model weights (~80 MB+
per model) on first use. The conceptual cells below need only `numpy` + `scipy`; the real
Demucs cell is gated so the notebook runs end-to-end even without the heavy deps.

```bash
# Real install (heavy — torch + weights). Run in your environment, not required here.
pip install demucs                  # pulls torch, torchaudio
# CLI usage (simplest path):
python -m demucs --two-stems vocals song.mp3      # -> separated/htdemucs/song/{vocals,no_vocals}.wav
python -m demucs -n htdemucs_ft song.mp3          # all four stems, fine-tuned model
```

In [1]:
# Lightweight deps for the conceptual demo (CPU, no downloads).
%pip install -q numpy scipy  # no-op if already present

import numpy as np
from scipy.signal import stft, istft

print("numpy", np.__version__)

# Is the real Demucs stack available? (Gates the last cell.)
try:
    import demucs  # noqa: F401
    HAVE_DEMUCS = True
except ImportError:
    HAVE_DEMUCS = False
print("demucs available:", HAVE_DEMUCS)

Note: you may need to restart the kernel to use updated packages.


numpy 2.5.0
demucs available: False


## 5. Worked Examples

### Example 1 — Why separation is possible: ideal time–frequency masking

We build a toy "mix" of two synthetic sources (a low tone ≈ bass, a high tone ≈ hi-hat),
move to the STFT domain, compute the **ideal ratio mask** for source A from the ground
truth, and apply it to the mix. This is the masking principle Demucs generalizes — the
real model *predicts* such a mask (and a waveform) instead of being handed the truth.

In [2]:
sr = 16000
t = np.linspace(0, 1.0, sr, endpoint=False)

# Two "sources" living mostly in different frequency bands.
src_a = 0.6 * np.sin(2 * np.pi * 220 * t)    # low tone  -> think bass
src_b = 0.6 * np.sin(2 * np.pi * 3000 * t)   # high tone -> think hi-hat
mix = src_a + src_b

# Move to time-frequency.
f, frames, Zmix = stft(mix, fs=sr, nperseg=1024)
_, _, Za = stft(src_a, fs=sr, nperseg=1024)
_, _, Zb = stft(src_b, fs=sr, nperseg=1024)

# Ideal Ratio Mask: fraction of each bin's energy that belongs to source A.
mask_a = np.abs(Za) / (np.abs(Za) + np.abs(Zb) + 1e-8)

# Apply the mask to the MIX and invert back to a waveform.
_, est_a = istft(mask_a * Zmix, fs=sr, nperseg=1024)
est_a = est_a[: len(src_a)]

print(f"mix waveform: {mix.shape}, spectrogram: {Zmix.shape} (freq x frames)")
print(f"mask values in [{mask_a.min():.2f}, {mask_a.max():.2f}]  "
      f"(near 1 where A dominates, near 0 where B dominates)")

mix waveform: (16000,), spectrogram: (513, 33) (freq x frames)
mask values in [0.00, 1.00]  (near 1 where A dominates, near 0 where B dominates)


### Example 2 — Scoring the separation with SI-SDR

The mix is a terrible estimate of source A (lots of B leaks in). After masking, the
estimate should score far higher. **SI-SDR** in dB is the standard separation metric —
the same number you'll see reported on Demucs benchmarks (real `htdemucs` lands around
**7–9 dB** SDR averaged over the MUSDB18 stems; our toy bands separate near-perfectly).

In [3]:
def si_sdr(reference, estimate, eps=1e-8):
    """Scale-invariant signal-to-distortion ratio (dB). Higher is better."""
    reference = reference - reference.mean()
    estimate = estimate - estimate.mean()
    alpha = np.dot(estimate, reference) / (np.dot(reference, reference) + eps)
    target = alpha * reference          # best-scaled projection onto the reference
    noise = estimate - target           # everything that isn't source A
    return 10 * np.log10((np.sum(target ** 2) + eps) / (np.sum(noise ** 2) + eps))

print(f"SI-SDR  mix      vs source A : {si_sdr(src_a, mix):6.1f} dB   (before separation)")
print(f"SI-SDR  estimate vs source A : {si_sdr(src_a, est_a):6.1f} dB   (after masking)")
print("\nMasking lifts SI-SDR by tens of dB here because the bands barely overlap;")
print("real music overlaps heavily, which is exactly why a learned model is needed.")

SI-SDR  mix      vs source A :   -0.0 dB   (before separation)
SI-SDR  estimate vs source A :   47.0 dB   (after masking)

Masking lifts SI-SDR by tens of dB here because the bands barely overlap;
real music overlaps heavily, which is exactly why a learned model is needed.


### Example 3 — The real Demucs call shape (gated)

This is the canonical pattern for running `htdemucs` in Python. It is **gated** behind
both the `demucs` import *and* a `DEMUCS_AUDIO` environment variable (path to an audio
file), so a fresh kernel without torch/weights still executes the notebook. Set
`DEMUCS_AUDIO=/path/to/song.mp3` and install `demucs` to run it for real — or just use
the CLI from the Setup section, which is the easiest path.

In [4]:
import os

def separate_with_demucs(path):
    """Return {stem_name: waveform tensor} for an audio file using htdemucs."""
    import torch, torchaudio
    from demucs.pretrained import get_model
    from demucs.apply import apply_model

    model = get_model("htdemucs")          # downloads ~80 MB the first time
    model.cpu().eval()

    wav, file_sr = torchaudio.load(path)   # (channels, samples)
    wav = torchaudio.functional.resample(wav, file_sr, model.samplerate)
    ref = wav.mean(0)
    wav = (wav - ref.mean()) / (ref.std() + 1e-8)   # Demucs expects normalized input

    with torch.no_grad():
        # apply_model handles chunking; input is (batch, channels, samples).
        sources = apply_model(model, wav[None], device="cpu", progress=False)[0]
    sources = sources * ref.std() + ref.mean()      # undo normalization
    return dict(zip(model.sources, sources))         # keys: drums, bass, other, vocals

audio = os.getenv("DEMUCS_AUDIO")
if HAVE_DEMUCS and audio:
    stems = separate_with_demucs(audio)
    for name, wave in stems.items():
        print(f"{name:7s} -> tensor shape {tuple(wave.shape)}")
else:
    print(f"demucs installed: {HAVE_DEMUCS} | DEMUCS_AUDIO set: {bool(audio)}")
    print("Skipping real separation. Set both to run; see separate_with_demucs() above.")
    print("CLI equivalent:  python -m demucs -n htdemucs song.mp3")

demucs installed: False | DEMUCS_AUDIO set: False
Skipping real separation. Set both to run; see separate_with_demucs() above.
CLI equivalent:  python -m demucs -n htdemucs song.mp3


## 6. Gotchas & Pitfalls

- **It's offline and slow.** A full song takes seconds-to-minutes on GPU and several
  minutes on CPU. Not for real-time. Use `segment` to cap memory; large `shifts` multiply
  runtime.
- **Stereo matters.** Demucs is trained on stereo; feeding mono works but quality drops.
  Don't downmix to mono before separating if you can avoid it.
- **`htdemucs_ft` is ~4× slower** than `htdemucs` (it runs four fine-tuned sub-models).
  Use it for final renders, the plain model for iteration.
- **Sample rate.** Resample to the model's `samplerate` (44.1 kHz) — `apply_model` does
  *not* resample for you. Mismatched SR silently wrecks output.
- **Normalize input.** The reference pipeline subtracts mean / divides by std before
  inference and reverses it after. Skipping this degrades quality.
- **Bleed and artifacts are expected.** Even SOTA leaves vocal breath in `other`, cymbal
  wash in `drums`, etc. There is no perfect un-mix; manage expectations.
- **The 4th stem is a junk drawer.** `other` = everything that isn't drums/bass/vocals
  (guitars, keys, synths, strings). Use `htdemucs_6s` if you specifically need guitar/piano.
- **Memory on long tracks / GPU OOM.** Lower `segment` (e.g. 7.8s) or run on CPU.
- **Licensing of the *audio*** you separate is your responsibility — the model is MIT/
  research-friendly, but the songs you feed it usually aren't yours to redistribute.

## 7. When to Use vs Alternatives

| Need | Best pick | Why |
|---|---|---|
| **Best open-source music stems** | **Demucs (`htdemucs`/`htdemucs_ft`)** | SOTA quality, MIT-ish, active, 4–6 stems |
| Fastest "good enough" stems, TF-based | **Spleeter** (Deezer) | Very fast, lower quality, TensorFlow, 2/4/5-stem |
| Open-source toolkit / research SDK | **Asteroid**, **SpeechBrain** | Recipes for custom training, speech *and* music |
| Highest quality, no ops | **Hosted services** (LALAL.AI, moises.ai, iZotope RX) | Pay per track; often beat OSS, zero setup |
| **Speech** denoise/dereverb, not music | **DeepFilterNet**, RNNoise, speech-enhancement nets | Tuned for voice, real-time options |
| Real-time / low latency | None of the above ideally | Demucs/Spleeter are offline; specialized streaming models only |

**Rules of thumb.** Default to **Demucs** for music. Drop to **Spleeter** when you need
to batch-process thousands of tracks fast and can tolerate more bleed. Use a **hosted
service** when quality is paramount and you'd rather pay than tune. Use a **speech**
enhancement model — not a music separator — when the source is a single voice.

## 8. Resources

- **Demucs repo (docs, model zoo, CLI):** https://github.com/adefossez/demucs
- **Hybrid Transformer Demucs paper (Rouard et al., 2022):** https://arxiv.org/abs/2211.08553
- **Original Demucs paper (Défossez et al., 2019):** https://arxiv.org/abs/1911.13254
- **MUSDB18 benchmark dataset:** https://sigsep.github.io/datasets/musdb.html
- **Spleeter (the fast alternative):** https://github.com/deezer/spleeter
- **Asteroid (separation toolkit):** https://github.com/asteroid-team/asteroid
- **SDR/SI-SDR explained ("SDR – half-baked or well done?"):** https://arxiv.org/abs/1811.02508